# Exercice 12 - Operations DataFrame avancees

## Objectifs
- Maitriser les operations de pivot
- Utiliser les User Defined Functions (UDF)
- Travailler avec les types complexes (arrays, maps)
- Optimiser les performances des DataFrames

---

## 1. Configuration

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, IntegerType, ArrayType, MapType, StructType, StructField
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Operations DataFrame Avancees") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .getOrCreate()

print("Spark pret")

In [ ]:
# Charger les donnees
jdbc_url = "jdbc:postgresql://postgres:5432/app"
jdbc_properties = {
    "user": "postgres",
    "password": "postgres",
    "driver": "org.postgresql.Driver"
}

df_orders = spark.read.jdbc(url=jdbc_url, table="orders", properties=jdbc_properties)
df_order_details = spark.read.jdbc(url=jdbc_url, table="order_details", properties=jdbc_properties)
df_products = spark.read.jdbc(url=jdbc_url, table="products", properties=jdbc_properties)
df_customers = spark.read.jdbc(url=jdbc_url, table="customers", properties=jdbc_properties)

print("Donnees chargees")

## 2. Pivot et Unpivot

```
PIVOT : Lignes -> Colonnes

Avant:                          Apres:
+-------+------+-------+        +-------+--------+--------+--------+
| annee | mois | ventes|   -->  | annee | Jan    | Fev    | Mar    |
+-------+------+-------+        +-------+--------+--------+--------+
| 2023  | Jan  | 100   |        | 2023  | 100    | 200    | 150    |
| 2023  | Fev  | 200   |        | 2024  | 120    | 180    | 160    |
| 2023  | Mar  | 150   |        +-------+--------+--------+--------+
| 2024  | Jan  | 120   |
+-------+------+-------+
```

In [ ]:
# Preparer les donnees de ventes
df_ventes = df_order_details.join(df_orders.select("order_id", "order_date"), on="order_id")
df_ventes = df_ventes.withColumn(
    "montant",
    F.round(F.col("unit_price") * F.col("quantity"), 2)
).withColumn(
    "annee", F.year("order_date")
).withColumn(
    "mois", F.month("order_date")
)

# Agreger par annee et mois
df_mensuel = df_ventes.groupBy("annee", "mois").agg(
    F.round(F.sum("montant"), 2).alias("ca")
)

df_mensuel.orderBy("annee", "mois").show()

In [ ]:
# PIVOT : mois en colonnes
df_pivot = df_mensuel.groupBy("annee").pivot("mois").agg(F.first("ca"))

print("Tableau pivot par mois :")
df_pivot.orderBy("annee").show()

In [ ]:
# Pivot avec liste de valeurs (plus performant)
mois_list = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
df_pivot_opt = df_mensuel.groupBy("annee").pivot("mois", mois_list).agg(F.first("ca")).fillna(0)

df_pivot_opt.show()

In [ ]:
# UNPIVOT : Colonnes -> Lignes
# Spark n'a pas de fonction unpivot native, on utilise stack()
colonnes_mois = [f"`{i}` as mois_{i}" for i in range(1, 13)]

df_unpivot = df_pivot_opt.select(
    "annee",
    F.expr(f"stack(12, {', '.join([f'{i}, `{i}`' for i in range(1, 13)])}) as (mois, ca)")
).filter(F.col("ca").isNotNull())

print("Apres unpivot :")
df_unpivot.orderBy("annee", "mois").show()

## 3. User Defined Functions (UDF)

In [ ]:
from pyspark.sql.functions import udf

# Definir une fonction Python
def categoriser_prix(prix):
    """Categorise un prix en niveau"""
    if prix is None:
        return "Inconnu"
    elif prix < 10:
        return "Economique"
    elif prix < 25:
        return "Standard"
    elif prix < 50:
        return "Premium"
    else:
        return "Luxe"

# Enregistrer comme UDF
categoriser_prix_udf = udf(categoriser_prix, StringType())

# Utiliser l'UDF
df_products_cat = df_products.withColumn(
    "niveau_prix",
    categoriser_prix_udf(F.col("unit_price"))
)

df_products_cat.select("product_name", "unit_price", "niveau_prix").show(10)

In [ ]:
# UDF avec decorateur (plus lisible)
@udf(returnType=StringType())
def format_telephone(phone):
    """Formate un numero de telephone"""
    if phone is None:
        return None
    # Garder seulement les chiffres
    digits = ''.join(filter(str.isdigit, phone))
    return digits if len(digits) > 0 else None

df_customers.withColumn(
    "phone_clean",
    format_telephone(F.col("phone"))
).select("company_name", "phone", "phone_clean").show(10)

In [ ]:
# UDF retournant une structure complexe
@udf(returnType=ArrayType(StringType()))
def extraire_mots(texte):
    """Extrait les mots d'un texte"""
    if texte is None:
        return []
    return texte.split()

df_products.withColumn(
    "mots_nom",
    extraire_mots(F.col("product_name"))
).select("product_name", "mots_nom").show(5, truncate=False)

## 4. Pandas UDF (plus performant)

In [ ]:
import pandas as pd
from pyspark.sql.functions import pandas_udf

# Pandas UDF (vectorisee, beaucoup plus rapide)
@pandas_udf("double")
def calculer_remise(prix: pd.Series, quantite: pd.Series) -> pd.Series:
    """Calcule une remise progressive"""
    montant = prix * quantite
    remise = pd.Series([0.0] * len(montant))
    remise[montant >= 100] = 0.05
    remise[montant >= 500] = 0.10
    remise[montant >= 1000] = 0.15
    return remise

df_order_details.withColumn(
    "remise_calculee",
    calculer_remise(F.col("unit_price"), F.col("quantity"))
).select("order_id", "unit_price", "quantity", "remise_calculee").show(10)

## 5. Types complexes : Arrays

In [ ]:
# Creer un array des produits par commande
df_produits_commande = df_order_details.join(
    df_products.select("product_id", "product_name"),
    on="product_id"
).groupBy("order_id").agg(
    F.collect_list("product_name").alias("produits"),
    F.collect_set("product_id").alias("product_ids")
)

df_produits_commande.show(5, truncate=False)

In [ ]:
# Operations sur les arrays
df_array_ops = df_produits_commande.withColumn(
    "nb_produits",
    F.size("produits")
).withColumn(
    "premier_produit",
    F.element_at("produits", 1)
).withColumn(
    "produits_tries",
    F.sort_array("produits")
)

df_array_ops.select("order_id", "nb_produits", "premier_produit", "produits_tries").show(5, truncate=False)

In [ ]:
# Explode : Array -> Lignes multiples
df_explode = df_produits_commande.select(
    "order_id",
    F.explode("produits").alias("produit")
)

print("Apres explode :")
df_explode.show(10)

## 6. Types complexes : Maps

In [ ]:
# Creer une map produit -> quantite par commande
df_map = df_order_details.join(
    df_products.select("product_id", "product_name"),
    on="product_id"
).groupBy("order_id").agg(
    F.map_from_entries(
        F.collect_list(
            F.struct(F.col("product_name"), F.col("quantity"))
        )
    ).alias("produits_quantites")
)

df_map.show(3, truncate=False)

In [ ]:
# Acceder aux valeurs d'une map
df_map.withColumn(
    "toutes_cles",
    F.map_keys("produits_quantites")
).withColumn(
    "toutes_valeurs",
    F.map_values("produits_quantites")
).select("order_id", "toutes_cles", "toutes_valeurs").show(3, truncate=False)

## 7. Operations sur les Structs

In [ ]:
# Creer une structure imbriquee
df_struct = df_customers.select(
    "customer_id",
    "company_name",
    F.struct(
        F.col("address").alias("rue"),
        F.col("city").alias("ville"),
        F.col("postal_code").alias("code_postal"),
        F.col("country").alias("pays")
    ).alias("adresse_complete")
)

df_struct.printSchema()
df_struct.show(5, truncate=False)

In [ ]:
# Acceder aux champs d'une structure
df_struct.select(
    "company_name",
    F.col("adresse_complete.ville").alias("ville"),
    F.col("adresse_complete.pays").alias("pays")
).show(10)

## 8. Optimisation des performances

In [ ]:
# Broadcast join (pour petites tables)
from pyspark.sql.functions import broadcast

# La table categories est petite -> broadcast
df_categories = spark.read.jdbc(url=jdbc_url, table="categories", properties=jdbc_properties)

df_joined = df_products.join(
    broadcast(df_categories),
    on="category_id",
    how="left"
)

df_joined.show(5)

In [ ]:
# Cache pour les DataFrames reutilises
df_ventes_cached = df_ventes.cache()

# Premiere utilisation (materialise le cache)
print(f"Nombre de lignes : {df_ventes_cached.count()}")

# Utilisations suivantes (depuis le cache)
print(f"CA total : {df_ventes_cached.agg(F.sum('montant')).collect()[0][0]:.2f}")

In [ ]:
# Voir le plan d'execution
df_joined.explain()

In [ ]:
# Liberer le cache
df_ventes_cached.unpersist()

---

## Exercice

**Objectif** : Creer une UDF et un pivot

**Consigne** :
1. Creez une UDF qui categorise les pays en regions (Europe, Amerique, Asie, Autre)
2. Appliquez-la aux customers
3. Faites un pivot des ventes par region et par trimestre

A vous de jouer :

In [ ]:
# TODO: Creer l'UDF de categorisation par region
@udf(returnType=StringType())
def categoriser_region(pays):
    pass  # A completer

In [ ]:
# TODO: Appliquer aux customers et faire le pivot

---

## Resume

Dans ce notebook, vous avez appris :
- Comment faire un **pivot** (lignes vers colonnes)
- Comment creer des **UDF** Python et Pandas
- Comment travailler avec les **arrays** et **maps**
- Comment creer et utiliser des **structs**
- Comment **optimiser** les performances (broadcast, cache)

### Prochaine etape
Dans le prochain notebook, nous approfondirons les jointures Spark.